# CZI image basics: demo and tests

This notebook runs each function in `czi_basics.py` on the sample file in `data/`
and checks the results with `assert` statements. If every cell runs without an
error, the code works.

The sample, `data/sample_nuclei_foci.czi`, is a small crop of a real confocal image
of irradiated cells: channel 0 is **DAPI** (nuclei) and channel 1 is **GFP**
(DNA damage repair foci), with 7 focal planes (Z).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from czi_basics import load_czi, max_projection, save_image, to_8bit

SAMPLE = Path("data/sample_nuclei_foci.czi")
OUT = Path("outputs")

## 1. `load_czi`: read the file

In [ ]:
stack, meta = load_czi(SAMPLE)
meta

In [ ]:
assert stack.shape == (2, 7, 512, 512)          # C, Z, Y, X
assert stack.dtype == np.uint16                 # raw microscope data is 16-bit
assert meta["channel_names"] == ["DAPI", "GFP"]
assert abs(meta["pixel_size_um"]["X"] - 0.1095) < 1e-3
print("load_czi OK")

## 2. `max_projection`: collapse Z

In [ ]:
mip = max_projection(stack)

assert mip.shape == (2, 512, 512)
assert (mip >= stack[:, 3]).all()               # never darker than any single plane
assert (mip == stack.max(axis=1)).all()
print("max_projection OK")

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(stack[1, 3], cmap="gray"); axes[0].set_title("GFP, single plane (z=3)")
axes[1].imshow(mip[1], cmap="gray");      axes[1].set_title("GFP, max projection")
for ax in axes: ax.axis("off")
plt.show()

## 3. `to_8bit`: 16-bit to 8-bit grayscale

In [ ]:
dapi8 = to_8bit(mip[0])
gfp8 = to_8bit(mip[1], low_pct=1, high_pct=99.9)

for img in (dapi8, gfp8):
    assert img.dtype == np.uint8
    assert img.min() == 0 and img.max() == 255  # full contrast range is used

# edge cases: a flat image and a tiny hand-made example
assert (to_8bit(np.full((4, 4), 7)) == 0).all()
assert to_8bit(np.array([[0, 50], [100, 100]]), 0, 100).tolist() == [[0, 128], [255, 255]]
print("to_8bit OK")

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(dapi8, cmap="gray"); axes[0].set_title("DAPI (8-bit)")
axes[1].imshow(gfp8, cmap="gray");  axes[1].set_title("GFP foci (8-bit)")
for ax in axes: ax.axis("off")
plt.show()

## 4. `save_image`: write PNG and TIFF

In [ ]:
from PIL import Image

written = []
for name, img in zip(meta["channel_names"], (dapi8, gfp8)):
    written.append(save_image(img, OUT / f"sample_{name}.png"))
written.append(save_image(mip[1], OUT / "sample_GFP_16bit.tif"))  # keeps full 16-bit data

for p in written:
    assert p.exists()
    print(p, Image.open(p).size, Image.open(p).mode)

assert (np.array(Image.open(OUT / "sample_GFP.png")) == gfp8).all()        # PNG is lossless
assert (np.array(Image.open(OUT / "sample_GFP_16bit.tif")) == mip[1]).all()  # TIFF keeps 16 bits
print("save_image OK")

## Putting it together

The whole conversion is four lines, so it is easy to loop over a folder of `.czi`
files:

In [ ]:
def czi_to_pngs(czi_path, out_dir):
    stack, meta = load_czi(czi_path)
    mip = max_projection(stack)
    return [save_image(to_8bit(ch), Path(out_dir) / f"{Path(czi_path).stem}_{name}.png")
            for name, ch in zip(meta["channel_names"], mip)]

czi_to_pngs(SAMPLE, OUT)

Bonus: a two-colour overlay (DAPI in blue, foci in green), the usual way these
images are shown in figures.

In [ ]:
rgb = np.zeros((*gfp8.shape, 3), dtype=np.uint8)
rgb[..., 1] = gfp8
rgb[..., 2] = dapi8
try:  # save_image only takes 2D (single-channel) images, so RGB is rejected
    save_image(rgb, OUT / "overlay.png")
except ValueError as err:
    print("expected error:", err)

plt.figure(figsize=(5, 5)); plt.imshow(rgb); plt.axis("off"); plt.title("DAPI + GFP")
plt.show()
print("All checks passed.")